In [67]:
NAME - KHUSHBOO GUPTA
DATA SCIENCE/ DATA ANALYTICS BATCH

In [ ]:
# SpendDNA: Rahul Sharma Transaction Analysis
Minor Project 2

This project analyses Rahul Sharma's 6-month bank transaction history and builds a SpendDNA profile using Python, NumPy and Pandas.

## Features Implemented
1. Data Cleaning
2. Vendor Extraction
3. Category Mapping
4. Spending Overview
5. Monthly Trend Analysis
6. Time-of-Day & Day-of-Week Analysis
7. Anomaly Detection
8. Spending Archetype Detection
9. Final SpendDNA Report

# **FEATURE 1 : TRANSACTION PARSER**

In [68]:
#-------IMPORT LIBRARIES------#

import pandas as pd
import numpy as np

In [69]:
#------LOADING DATA SET-------#

df = pd.read_csv("Data set for DADS June.csv")

print("Original dataset shape:", df.shape)
df.head()

Original dataset shape: (1328, 8)


,Date,Time,Description,Type,Amount,Balance,Mode,Ref
0,2024-01-01,03:11,AMAZON SELLER SVCS,Debit,₹2462,678275.0,UPI,TXN190872
1,01-Jan-24,05:44,BHIM-BMTC,DR,50.00,681007.0,UPI,TXN143064
2,01-Jan-24,09:35,NEFT-TECHCRUSH LABS-SALARY MAY24,CR,₹84728,484728.0,NEFT,TXN246316
3,2024-01-01,14:07,UPI-AMAN-8934@OKAXIS,Debit,₹1828,-748745.0,UPI,TXN569226
4,01 Jan 2024,14:23,BHIM-BLINKIT,Debit,270.00,680737.0,UPI,TXN968962


In [70]:
#------CREATE WORKING COPY------#

df_clean = df.copy()
print("df_clean reset shape:", df_clean.shape)

df_clean reset shape: (1328, 8)


In [71]:
#------CHECKING RAW DATASET------#

print("column names:")
print(df_clean.columns.tolist())

print("n/ initial missing values:")
print(df_clean.isna().sum())

column names:
['Date', 'Time', 'Description', 'Type', 'Amount', 'Balance', 'Mode', 'Ref']
n/ initial missing values:
Date           0
Time           0
Description    0
Type           0
Amount         0
Balance        0
Mode           0
Ref            0
dtype: int64


In [72]:
#------PARSING DATE COLUMN------#
dt_formats = ['%Y-%m-%d', '%d/%m/%y', '%d-%b-%y', '%d %b %Y']

parsed_dt_list = []

for value in df_clean['Date']:
    dt_txt = str(value).strip()
    parsed_value = pd.NaT

    for fmt in dt_formats:
        try:
            parsed_value = pd.to_datetime(dt_txt, format=fmt)
            break
        except ValueError:
            continue

    parsed_dt_list.append(parsed_value)

df_clean['Date'] = pd.Series(parsed_dt_list)

wrong_dates = df_clean['Date'].isna().sum()

print("Unparseable dates:", wrong_dates)
print("Date column dtype:", df_clean['Date'].dtype)

Unparseable dates: 0
Date column dtype: datetime64[ns]


In [73]:
#-----CLEANING THE AMOUNT COLUMN------#

clean_ammts = []

for value in df_clean['Amount']:
    ammt_txt = str(value).strip()
    ammt_txt = ammt_txt.replace('₹', '')
    ammt_txt = ammt_txt.replace('Rs.', '')
    ammt_txt = ammt_txt.replace(',', '')
    ammtt_txt = ammt_txt.strip()
    clean_ammts.append(ammt_txt)

df_clean['Amount'] = pd.to_numeric(clean_ammts, errors='coerce')

wrong_amounts = df_clean['Amount'].isna().sum()

print("Unparseable amounts:", wrong_amounts)
print("Amount column dtype:", df_clean['Amount'].dtype)

Unparseable amounts: 0
Amount column dtype: float64


In [74]:
#------STANDARDIZING THE TYPE COUMN------#

clean_categories = []

for value in df_clean['Type']:
    type_txt = str(value).strip().lower()

    if type_txt == 'dr' or type_txt == 'debit':
        clean_categories.append('debit')
    elif type_txt == 'cr' or type_txt == 'credit':
        clean_categories.append('credit')
    else:
        clean_categories.append(np.nan)

df_clean['Type'] = clean_categories

wrong_types = df_clean['Type'].isna().sum()

print("Unmapped type values:", wrong_types)

Unmapped type values: 0


In [75]:
#------CLEANING THE MODE COLUMN------#

is_clean_modes = []

for value in df_clean['Mode']:
    mode_txt = str(value).strip()

    if mode_txt == '':
        is_clean_modes.append(np.nan)
    else:
        is_clean_modes.append(mode_txt)

df_clean['Mode'] = is_clean_modes

print("Missing Mode values after cleaning:", df_clean['Mode'].isna().sum())

Missing Mode values after cleaning: 0


In [76]:
#-----REMOVING DUPLICATE ROWS-----#

rows_previous = df_clean.shape[0]

df_clean = df_clean.drop_duplicates()

rows_following = df_clean.shape[0]
duplicates_dropped = rows_previous - rows_following

print("Rows before removing duplicates:", rows_previous)
print("Rows after removing duplicates:", rows_following)
print("Duplicates removed:", duplicates_dropped)

Rows before removing duplicates: 1328
Rows after removing duplicates: 1310
Duplicates removed: 18


In [77]:
#------DROPPING ROWS WHERE PARSING FAILED------#

df_clean = df_clean.dropna(subset=['Date', 'Amount', 'Type']).copy()

print("Shape after dropping invalid rows:", df_clean.shape)

Shape after dropping invalid rows: (1310, 8)


In [78]:
#------FINAL PARSER SUMMARY------#

approved_transactions_record = df_clean.shape[0]

print(
    f"Parsed {approved_transactions_record} transactions across 6 months. "
    f"Dropped {duplicates_dropped} duplicates. "
    f"{wrong_amounts} unparseable amounts, {wrong_dates} unparseable dates."
)

Parsed 1310 transactions across 6 months. Dropped 18 duplicates. 0 unparseable amounts, 0 unparseable dates.


# **FEATURE 2 : VENDOR EXTRACTOR**

In [79]:
#------CHECKING UNIQUE TRANSACTION DESCRIPTION------#

unique_desc_set = df_clean['Description'].dropna().unique()

print("Number of unique descriptions:", len(unique_desc_set))
print("\nFirst 50 unique descriptions:\n")

for desc in unique_desc_set[:50]:
    print(desc)

Number of unique descriptions: 283

First 50 unique descriptions:

AMAZON SELLER SVCS
BHIM-BMTC
NEFT-TECHCRUSH LABS-SALARY MAY24
UPI-AMAN-8934@OKAXIS
BHIM-BLINKIT
BHIM ZEPTO
UPI-UBER-2426@HDFCBANK
POS SWIGGY BANGALORE
UPI-GROWWPAY@HDFCBANK
OLA ELECTRIC
BMS MOVIE TICKETS
POS OLA-PRIME
SWIGGY-INSTAMART
UPI-STARBUCKS@AXIS
UPI-THIRDWAVE@OKAXIS
ANI Technologies
BMTC BUS PASS
POS TRUFFLES
FLIPKART INDIA
POS SWIGGY-RESTAURANT
GROFERS INDIA P L
POS UBER BANGALORE
BANGALORE ELEC SUPPLY
TWC INDIA
UPI-BESCOM-BILL@HDFCBANK
UPI-AMAN-0816@OKAXIS
ROPPEN TRANSPORTATION
OLA CABS
POS ZOMATO
UPI-AMAZONPAY@HDFCBANK
POS BLINKIT
IMPS-RENT-LANDLORD-75500265
ZOMATO MEDIA P L
UPI-ANKIT-6430@OKAXIS
UPI-OLACABS@HDFCBANK
UPI-JIORECHARGE@PAYTM
UPI-CCD@HDFCBANK
Swiggy*Order
INSTAMART BANGALORE
UPI-ZOMATO-LIMITED@PAYTM
AVENUE SUPERMARTS
POS HP PETROL STATION
UPI-VIKAS-6060@OKAXIS
POS BANGALORE RESTAURANT
UPI-ZERODHA-COIN@AXIS
BHIM SWIGGY
UPI-BOOKMYSHOW@HDFCBANK
BLINKIT BANGALORE
POS ZEPTO BANGALORE
UPI-DMART@OKAXIS


In [80]:
#------CREATING VENDOR DICTIONARY------#

vendor_lookup = {

    # Food Delivery
    'Swiggy': ['SWIGGY', 'BUNDL', 'INSTAMART'],
    'Zomato': ['ZOMATO'],

    # Quick Commerce
    'Blinkit': ['BLINKIT', 'GROFERS'],
    'Zepto': ['ZEPTO'],

    # E-commerce
    'Amazon': ['AMAZON', 'AMAZONPAY', 'AMAZON IN', 'AMAZONIN'],
    'Flipkart': ['FLIPKART', 'FKART'],

    # Transport
    'Uber': ['UBER', 'ANI TECHNOLOGIES'],
    'Ola': ['OLA'],
    'Rapido': ['RAPIDO', 'ROPPEN'],
    'BMTC': ['BMTC'],

    # Utilities
    'BESCOM': ['BESCOM', 'BANGALORE ELEC'],
    'Water Bill': ['BWSSB'],

    # Entertainment
    'BookMyShow': ['BOOKMYSHOW', 'BMS', 'BIGTREE'],

    # Cafes
    'Starbucks': ['STARBUCKS'],
    'Third Wave Coffee': ['THIRDWAVE', 'TWC'],
    'Cafe Coffee Day': ['COFFEE DAY'],

    # Restaurants
    'Truffles': ['TRUFFLES'],
    'Restaurant': ['RESTAURANT'],

    # Grocery
    'DMart': ['DMART', 'AVENUE'],
    'BigBasket': ['BIGBASKET', 'KIRANAKART'],

    # Investments
    'Groww': ['GROWW'],
    'Zerodha': ['ZERODHA'],

    # Fuel
    'HP Petrol': ['HP PETROL'],

    # Recharge
    'Jio': ['JIO'],
    'Airtel': ['AIRTEL', 'BHARTI AIRTEL'],
    'Vodafone Idea': ['VODAFONE', 'VI POSTPAID'],

    # Shopping
    'Myntra': ['MYNTRA'],
    'Nykaa': ['NYKAA', 'FSN E-COMMERCE'],

    # OTT
    'Netflix': ['NETFLIX'],
    'Spotify': ['SPOTIFY'],
    'Disney Hotstar': ['DISNEY HOTSTAR'],
    'Amazon Prime': ['AMZN PRIME'],

    # Income
    'Salary': ['SALARY']
}

In [81]:
#-------VENDOR EXTRACTION-------#


def extracted_vendor_data(description):

    desc = str(description).strip().upper()

    # Cash Withdrawal
    if 'ATM-WDL' in desc or 'ATM WDL' in desc:
        return 'Cash Withdrawal'

    # Salary
    if 'SALARY' in desc:
        return 'Salary'

    # Rent
    if 'RENT' in desc or 'LANDLORD' in desc:
        return 'Rent'

    # P2P Transfer
    if desc.startswith('UPI-'):

        known_vendor_names = [
            'SWIGGY', 'BUNDL', 'INSTAMART',
            'ZOMATO',
            'BLINKIT', 'GROFERS',
            'ZEPTO',
            'AMAZON', 'AMAZONPAY',
            'UBER',
            'OLA',
            'BMTC',
            'BESCOM',
            'BOOKMYSHOW', 'BMS',
            'STARBUCKS',
            'THIRDWAVE', 'TWC',
            'TRUFFLES',
            'DMART', 'AVENUE',
            'HP',
            'JIO',
            'GROWW',
            'ZERODHA',
            'MYNTRA',
            'NYKAA',
            'NETFLIX',
            'SPOTIFY',
            'DISNEY',
            'RAPIDO',
            'AIRTEL',
            'VODAFONE',
            'BWSSB'
        ]

        found = False

        for word in known_vendor_names:
            if word in desc:
                found = True
                break

        if not found:
            return 'P2P Transfer'

    # Vendor Dictionary Lookup
    for vendor_name, keywords in vendor_lookup.items():

        for keyword in keywords:

            if keyword.upper() in desc:
                return vendor_name

    # POS Transactions not matched
    if desc.startswith('POS'):
        return 'Card Payment'

    return 'Uncategorised'

In [82]:
df_clean['vendor_cleaned_names'] = df_clean['Description'].apply(extracted_vendor_data)

In [83]:
#------APPLYING VENDOR EXTRACTION------#

df_clean['vendor_cleaned_names'] = df_clean['Description'].apply(extracted_vendor_data)

print("vendor_cleaned_names column created successfully.\n")
print(df_clean[['Description', 'vendor_cleaned_names']].head(15))

vendor_cleaned_names column created successfully.

                         Description vendor_cleaned_names
0                 AMAZON SELLER SVCS               Amazon
1                          BHIM-BMTC                 BMTC
2   NEFT-TECHCRUSH LABS-SALARY MAY24               Salary
3               UPI-AMAN-8934@OKAXIS         P2P Transfer
4                       BHIM-BLINKIT              Blinkit
5                         BHIM ZEPTO                Zepto
6             UPI-UBER-2426@HDFCBANK                 Uber
7                       BHIM-BLINKIT              Blinkit
8               POS SWIGGY BANGALORE               Swiggy
9              UPI-GROWWPAY@HDFCBANK                Groww
10                      OLA ELECTRIC                  Ola
11                 BMS MOVIE TICKETS           BookMyShow
12                     POS OLA-PRIME                  Ola
13                  SWIGGY-INSTAMART               Swiggy
14                UPI-STARBUCKS@AXIS            Starbucks


In [84]:
#------VENDOR DISTRIBUTION CHECKING------#

print("Number of unique cleaned vendors:", df_clean['vendor_cleaned_names'].nunique())

print("\n Top 10 vendor counts:\n")
print(df_clean['vendor_cleaned_names'].value_counts().head(10))

Number of unique cleaned vendors: 38

 Top 10 vendor counts:

vendor_cleaned_names
Swiggy          243
Zomato          121
Uber             89
P2P Transfer     76
Amazon           70
Ola              69
Zepto            58
Rapido           55
Blinkit          55
Card Payment     46
Name: count, dtype: int64


In [85]:
#------VENDOR CLEANUP AUDIT------#

uncategorised_descs = df_clean.loc[
    df_clean['vendor_cleaned_names'] == 'Uncategorised',
    'Description'
].drop_duplicates()

print("Number of uncategorised descriptions:", len(uncategorised_descs))

print("\nUncategorised descriptions:\n")

for desc in uncategorised_descs:
    print(desc)

Number of uncategorised descriptions: 3

Uncategorised descriptions:

AMZN-INTPYMT
INNOVATIVE RETAIL
STAR INDIA PVT LTD


In [86]:
#-----FINAL CHECK-----#

print("Number of unique cleaned vendors:", df_clean['vendor_cleaned_names'].nunique())

print("\n Top 10 vendor counts:\n")
print(df_clean['vendor_cleaned_names'].value_counts().head(10))

Number of unique cleaned vendors: 38

 Top 10 vendor counts:

vendor_cleaned_names
Swiggy          243
Zomato          121
Uber             89
P2P Transfer     76
Amazon           70
Ola              69
Zepto            58
Rapido           55
Blinkit          55
Card Payment     46
Name: count, dtype: int64


# **FEATURE 3 : CATEGORY TAGGER**

In [87]:
#-------CREATING DICTIONARY-------#


category_lookup = {

    # Food Delivery
    'Swiggy': 'Food Delivery',
    'Zomato': 'Food Delivery',

    # Quick Commerce
    'Blinkit': 'Quick Commerce',
    'Zepto': 'Quick Commerce',

    # E-commerce
    'Amazon': 'E-commerce',
    'Amazon Prime': 'Subscriptions',
    'Flipkart': 'E-commerce',
    'Myntra': 'E-commerce',
    'Nykaa': 'E-commerce',

    # Transport
    'Uber': 'Transport',
    'Ola': 'Transport',
    'Rapido': 'Transport',
    'BMTC': 'Transport',
    'Bus / Metro': 'Transport',

    # Cafes
    'Starbucks': 'Cafe',
    'Third Wave Coffee': 'Cafe',
    'Cafe Coffee Day': 'Cafe',

    # Restaurants
    'Truffles': 'Restaurants',
    'Restaurant': 'Restaurants',

    # Utilities
    'BESCOM': 'Utilities',
    'Electricity': 'Utilities',
    'Water Bill': 'Utilities',
    'Airtel': 'Utilities',
    'Vodafone Idea': 'Utilities',
    'Jio': 'Utilities',

    # Groceries
    'DMart': 'Groceries',
    'BigBasket': 'Groceries',

    # Investments
    'Groww': 'Investments',
    'Zerodha': 'Investments',

    # Fuel
    'HP Petrol': 'Fuel',

    # Entertainment
    'BookMyShow': 'Entertainment',
    'Disney Hotstar': 'Entertainment',
    'Netflix': 'Entertainment',
    'Spotify': 'Entertainment',
    'Star India': 'Entertainment',

    # Income / Transfers
    'Salary': 'Income',
    'Rent': 'Rent',
    'P2P Transfer': 'Personal Transfer',
    'Cash Withdrawal': 'Cash Withdrawal',

    # Others
    'Card Payment': 'Card Payment'
}

In [88]:
#-------MAPPING VENDORS-------#


df_clean['category'] = df_clean['vendor_cleaned_names'].map(category_lookup)

df_clean['category'] = df_clean['category'].fillna('Others')

print("Category column created successfully.\n")

print(df_clean[['vendor_cleaned_names','category']].head(15))

Category column created successfully.

   vendor_cleaned_names           category
0                Amazon         E-commerce
1                  BMTC          Transport
2                Salary             Income
3          P2P Transfer  Personal Transfer
4               Blinkit     Quick Commerce
5                 Zepto     Quick Commerce
6                  Uber          Transport
7               Blinkit     Quick Commerce
8                Swiggy      Food Delivery
9                 Groww        Investments
10                  Ola          Transport
11           BookMyShow      Entertainment
12                  Ola          Transport
13               Swiggy      Food Delivery
14            Starbucks               Cafe


In [89]:
#-------CATEGORY DISTRIBUTION-------#


print("Missing category values:",
      df_clean['category'].isna().sum())

print("\nTop Categories:\n")

print(df_clean['category'].value_counts())

Missing category values: 0

Top Categories:

category
Food Delivery        364
Transport            250
E-commerce           143
Quick Commerce       113
Cafe                  79
Personal Transfer     76
Card Payment          46
Utilities             40
Groceries             40
Entertainment         37
Restaurants           34
Investments           23
Others                22
Cash Withdrawal       17
Fuel                   9
Income                 6
Rent                   6
Subscriptions          5
Name: count, dtype: int64


In [90]:
#-------VERIFYING MISSING CATEGORIES-------#

print("Number of Categories:",
      df_clean['category'].nunique())

print("\nCategory List:\n")

for category in sorted(df_clean['category'].unique()):
    print(category)

Number of Categories: 18

Category List:

Cafe
Card Payment
Cash Withdrawal
E-commerce
Entertainment
Food Delivery
Fuel
Groceries
Income
Investments
Others
Personal Transfer
Quick Commerce
Rent
Restaurants
Subscriptions
Transport
Utilities


# **FEATURE 4 : *SPENDING OVERVIEW*

In [91]:
#------CALCULATING FINANCIAL METRICS------#


tot_credits = df_clean.loc[df_clean['Type'] == 'credit', 'Amount'].sum()

tot_debits = df_clean.loc[df_clean['Type'] == 'debit', 'Amount'].sum()

net_rate_savings = tot_credits - tot_debits

rate_savings = (net_rate_savings / tot_credits) * 100


print(f"Total Credits      : ₹{tot_credits:,.2f}")
print(f"Total Debits       : ₹{tot_debits:,.2f}")
print(f"Net Savings        : ₹{net_rate_savings:,.2f}")
print(f"Savings Rate       : {rate_savings:.2f}%")
print(f"Total Transactions : {len(df_clean)}")

Total Credits      : ₹509,774.00
Total Debits       : ₹1,678,901.00
Net Savings        : ₹-1,169,127.00
Savings Rate       : -229.34%
Total Transactions : 1310


In [92]:
#------VERIFY TRANSACTION TYPES------#

print("Transaction Count by Type:\n")
print(df_clean['Type'].value_counts())

print("\nTotal Amount by Type:\n")
print(df_clean.groupby('Type')['Amount'].sum())

if df_clean['Type'].value_counts().get('credit', 0) < 10:
    print("\nNote: This dataset contains very few credit transactions.")
    print("Therefore, the savings rate will differ from the sample expected output.")

Transaction Count by Type:

Type
debit     1304
credit       6
Name: count, dtype: int64

Total Amount by Type:

Type
credit     509774.0
debit     1678901.0
Name: Amount, dtype: float64

Note: This dataset contains very few credit transactions.
Therefore, the savings rate will differ from the sample expected output.


In [93]:
#-------TOP 5 VENDORS BY SPEND--------#

vendor_income_spend = (
    df_clean[df_clean['Type'] == 'debit']
    .groupby('vendor_cleaned_names')['Amount']
    .sum()
    .sort_values(ascending=False)
)

print("Top 5 Vendors by Spending:\n")
print(vendor_income_spend.head(5))

Top 5 Vendors by Spending:

vendor_cleaned_names
Amazon          299462.0
Zerodha         210000.0
P2P Transfer    149747.0
Rent            108000.0
Swiggy          104351.0
Name: Amount, dtype: float64


In [94]:
#------TOP 5 CATEGORIES BY SPENDING------#

category_income_spend = (
    df_clean[df_clean['Type'] == 'debit']
    .groupby('category')['Amount']
    .sum()
    .sort_values(ascending=False)
)

print("Top 5 Categories by Spending:\n")
print(category_income_spend.head(5))

Top 5 Categories by Spending:

category
E-commerce           495985.0
Investments          248160.0
Food Delivery        159667.0
Personal Transfer    149747.0
Rent                 108000.0
Name: Amount, dtype: float64


In [95]:
#----------EXECUTIVE SUMMARY----------#


print("=" * 50)
print("        SPENDING OVERVIEW REPORT")
print("=" * 50)

print(f"\nTotal Transactions : {len(df_clean)}")
print(f"Total Credits      : ₹{tot_credits:,.2f}")
print(f"Total Debits       : ₹{tot_debits:,.2f}")
print(f"Net Savings        : ₹{net_rate_savings:,.2f}")
print(f"Savings Rate       : {rate_savings:.2f}%")

print("\nTop Spending Category :", category_income_spend.idxmax())
print(f"Amount Spent          : ₹{category_income_spend.max():,.2f}")

print("\nTop Spending Vendor   :", vendor_income_spend.idxmax())
print(f"Amount Spent          : ₹{vendor_income_spend.max():,.2f}")

print("\n" + "=" * 50)

        SPENDING OVERVIEW REPORT

Total Transactions : 1310
Total Credits      : ₹509,774.00
Total Debits       : ₹1,678,901.00
Net Savings        : ₹-1,169,127.00
Savings Rate       : -229.34%

Top Spending Category : E-commerce
Amount Spent          : ₹495,985.00

Top Spending Vendor   : Amazon
Amount Spent          : ₹299,462.00



# **FEATURE 5 : MONTHLY TREND ANALYSIS**

In [96]:
#-------CREATING COLUMN OF MONTH-------#

df_clean['Month'] = df_clean['Date'].dt.strftime('%b')

month_sequence = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun']

print("Months in Dataset:\n")
print(df_clean['Month'].unique())

Months in Dataset:

['Jan' 'Feb' 'Mar' 'Apr' 'May' 'Jun']


In [97]:
#---------BUILDING MONTHLY SPENDING MATRIX---------#

monthly_pivot_list = (
    df_clean[df_clean['Type'] == 'debit']
    .pivot_table(
        values='Amount',
        index='category',
        columns='Month',
        aggfunc='sum',
        fill_value=0
    )
)

monthly_pivot_list = monthly_pivot_list.reindex(columns = month_sequence)

print("Monthly Spending Matrix:\n")

print(monthly_pivot_list)

Monthly Spending Matrix:

Month                  Jan      Feb       Mar      Apr      May       Jun
category                                                                 
Cafe                3332.0   3620.0    3637.0   4748.0   5467.0    4886.0
Card Payment       22270.0   7861.0   33094.0  11105.0  15166.0   12578.0
Cash Withdrawal     2000.0   5000.0    8000.0   5500.0   8000.0   17000.0
E-commerce         90082.0  86599.0  100992.0  49373.0  57233.0  111706.0
Entertainment       4030.0   3830.0    4016.0   3250.0    538.0    4293.0
Food Delivery      23692.0  26458.0   25918.0  30092.0  26598.0   26909.0
Fuel                4267.0    627.0    1727.0  10185.0      0.0    1089.0
Groceries          15852.0   5390.0    6205.0   6165.0   7845.0    7721.0
Investments        38432.0  15000.0   68644.0  54126.0  48628.0   23330.0
Others              6118.0   4803.0    4810.0   7661.0   4110.0    7926.0
Personal Transfer  18205.0  14242.0   15661.0  25292.0  45749.0   30598.0
Quick Commer

In [98]:
#-------BIGGEST MONTH-ON-MONTH GROWTH--------#

monthly_growth = (
    (monthly_pivot_list['Jun'] - monthly_pivot_list['Jan'])
    / monthly_pivot_list['Jan']
) * 100

monthly_growth = monthly_growth.replace([float('inf'), -float('inf')], 0)
monthly_growth = monthly_growth.fillna(0)

print("Growth (%) from January to June:\n")

print(monthly_growth.sort_values(ascending=False))

Growth (%) from January to June:

category
Cash Withdrawal      750.000000
Personal Transfer     68.074705
Cafe                  46.638655
Others                29.552141
E-commerce            24.004796
Food Delivery         13.578423
Entertainment          6.526055
Utilities              3.647483
Rent                   0.000000
Quick Commerce       -28.874437
Transport            -36.428896
Investments          -39.295379
Card Payment         -43.520431
Groceries            -51.293212
Restaurants          -58.842769
Fuel                 -74.478556
Subscriptions       -100.000000
dtype: float64


In [99]:
#--------BIGGEST GROWTH AND BIGGEST DECLINE---------#

highest_monthly_growth = monthly_growth.idxmax()
lowest_growth_rate = monthly_growth.idxmin()

print("Category with Highest Growth :",
      highest_monthly_growth)

print("Growth : {:.2f}%".format(monthly_growth.max()))

print()

print("Category with Biggest Decline :",
      lowest_growth_rate)

print("Decline : {:.2f}%".format(monthly_growth.min()))

Category with Highest Growth : Cash Withdrawal
Growth : 750.00%

Category with Biggest Decline : Subscriptions
Decline : -100.00%


In [100]:
#--------MONTHLY TREND SUMMARY--------#

print("=" * 60)
print("        MONTHLY TREND ANALYSIS")
print("=" * 60)

print("\nMonthly Spending Matrix:\n")
print(monthly_pivot_list)

print("\nHighest Growing Category :", highest_monthly_growth)
print("Growth                  : {:.2f}%".format(monthly_growth.max()))

print("\nBiggest Declining Category :", lowest_growth_rate)
print("Decline                    : {:.2f}%".format(monthly_growth.min()))

print("\n" + "=" * 60)

        MONTHLY TREND ANALYSIS

Monthly Spending Matrix:

Month                  Jan      Feb       Mar      Apr      May       Jun
category                                                                 
Cafe                3332.0   3620.0    3637.0   4748.0   5467.0    4886.0
Card Payment       22270.0   7861.0   33094.0  11105.0  15166.0   12578.0
Cash Withdrawal     2000.0   5000.0    8000.0   5500.0   8000.0   17000.0
E-commerce         90082.0  86599.0  100992.0  49373.0  57233.0  111706.0
Entertainment       4030.0   3830.0    4016.0   3250.0    538.0    4293.0
Food Delivery      23692.0  26458.0   25918.0  30092.0  26598.0   26909.0
Fuel                4267.0    627.0    1727.0  10185.0      0.0    1089.0
Groceries          15852.0   5390.0    6205.0   6165.0   7845.0    7721.0
Investments        38432.0  15000.0   68644.0  54126.0  48628.0   23330.0
Others              6118.0   4803.0    4810.0   7661.0   4110.0    7926.0
Personal Transfer  18205.0  14242.0   15661.0  25292.0

# **FEATURE 6 : TIME - OF - DAY PATTERNS**

In [101]:
#--------EXTRACTING HOUR FROM TIME---------#

df_clean['Hour'] = df_clean['Time'].str[:2].astype(int)

print("Hour column created successfully.\n")

print(df_clean[['Time', 'Hour']].head(10))

Hour column created successfully.

    Time  Hour
0  03:11     3
1  05:44     5
2  09:35     9
3  14:07    14
4  14:23    14
5  14:48    14
6  14:50    14
7  21:44    21
8  05:18     5
9  06:55     6


In [102]:
#--------CATEGORY Vs HOUR OF DAY MATRIX---------#

time_matrix = (
    df_clean[df_clean['Type'] == 'debit']
    .pivot_table(
        values='Amount',
        index='category',
        columns='Hour',
        aggfunc='count',
        fill_value=0
    )
)

time_matrix = time_matrix.reindex(columns=range(24), fill_value=0)

print("Category × Hour Matrix:\n")

print(time_matrix)

Category × Hour Matrix:

Hour               0   1   2   3   4   5   6   7   8   9   ...  14  15  16  \
category                                                   ...               
Cafe                2   2   0   1   0   0   1   2   4   5  ...   3   5   6   
Card Payment        3   0   0   2   1   2   0   0   3   1  ...   2   2   4   
Cash Withdrawal     0   0   0   0   0   0   0   0   2   3  ...   2   0   1   
E-commerce          8   5   4   9   7   4   7   3   4   7  ...   7   4   5   
Entertainment       1   2   1   3   7   2   3   3   2   0  ...   0   0   2   
Food Delivery       3   8   3   5   7   7   1   3   9  15  ...  10  18  11   
Fuel                0   0   0   0   0   0   0   0   1   2  ...   0   1   1   
Groceries           2   1   1   3   1   0   2   0   2   2  ...   3   0   4   
Investments         1   1   0   0   2   1   4   0   0   1  ...   0   0   0   
Others              1   0   0   5   2   1   0   0   0   3  ...   1   0   0   
Personal Transfer   0   2   3   1   2  

In [103]:
#--------PEAK SPENDING HOUR BY CATEGORY---------#

peak_duration_hr = time_matrix.idxmax(axis = 1)
peak_levels = time_matrix.max(axis = 1)

print("Peak Spending Hours by Category:\n")

for category in time_matrix.index:
    print(f"{category:20s} : {peak_duration_hr[category]:02d}:00 "
          f"({peak_levels[category]} transactions)")

Peak Spending Hours by Category:

Cafe                 : 10:00 (11 transactions)
Card Payment         : 20:00 (5 transactions)
Cash Withdrawal      : 09:00 (3 transactions)
E-commerce           : 11:00 (10 transactions)
Entertainment        : 04:00 (7 transactions)
Food Delivery        : 20:00 (43 transactions)
Fuel                 : 09:00 (2 transactions)
Groceries            : 16:00 (4 transactions)
Investments          : 10:00 (6 transactions)
Others               : 03:00 (5 transactions)
Personal Transfer    : 16:00 (7 transactions)
Quick Commerce       : 19:00 (13 transactions)
Rent                 : 13:00 (3 transactions)
Restaurants          : 18:00 (6 transactions)
Subscriptions        : 05:00 (3 transactions)
Transport            : 20:00 (23 transactions)
Utilities            : 14:00 (4 transactions)


In [104]:
#---------LIFESTYLE INSIGHTS----------#

print("=" * 65)
print("              TIME-OF-DAY SPENDING INSIGHTS")
print("=" * 65)

for category in time_matrix.index:

    tot = time_matrix.loc[category].sum()

    peak_duration_hr = time_matrix.loc[category].idxmax()

    max_transactions = time_matrix.loc[category].max()

    percent = (max_transactions / tot) * 100

    print(f"{category:20s} | Peak Hour: {peak_duration_hr:02d}:00 | "
          f"{percent:.1f}% of transactions")

              TIME-OF-DAY SPENDING INSIGHTS
Cafe                 | Peak Hour: 10:00 | 13.9% of transactions
Card Payment         | Peak Hour: 20:00 | 10.9% of transactions
Cash Withdrawal      | Peak Hour: 09:00 | 17.6% of transactions
E-commerce           | Peak Hour: 11:00 | 7.0% of transactions
Entertainment        | Peak Hour: 04:00 | 18.9% of transactions
Food Delivery        | Peak Hour: 20:00 | 11.8% of transactions
Fuel                 | Peak Hour: 09:00 | 22.2% of transactions
Groceries            | Peak Hour: 16:00 | 10.0% of transactions
Investments          | Peak Hour: 10:00 | 26.1% of transactions
Others               | Peak Hour: 03:00 | 22.7% of transactions
Personal Transfer    | Peak Hour: 16:00 | 9.2% of transactions
Quick Commerce       | Peak Hour: 19:00 | 11.5% of transactions
Rent                 | Peak Hour: 13:00 | 50.0% of transactions
Restaurants          | Peak Hour: 18:00 | 17.6% of transactions
Subscriptions        | Peak Hour: 05:00 | 60.0% of transaction

In [105]:
# Extended feature 6 : BONUS MARKS

#--------EXTRACTING DAY OF WEEK---------#
#--------CREATE DAY OF WEEK COLUMN--------#

df_clean['Day'] = df_clean['Date'].dt.day_name()

day_sequence = [
    'Monday',
    'Tuesday',
    'Wednesday',
    'Thursday',
    'Friday',
    'Saturday',
    'Sunday'
]

print("Day column created successfully.\n")

print(df_clean[['Date', 'Day']].head(10))

Day column created successfully.

        Date      Day
0 2024-01-01   Monday
1 2024-01-01   Monday
2 2024-01-01   Monday
3 2024-01-01   Monday
4 2024-01-01   Monday
5 2024-01-01   Monday
6 2024-01-01   Monday
7 2024-01-01   Monday
8 2024-01-02  Tuesday
9 2024-01-02  Tuesday


In [106]:
#-------DAY OF WEEK SPENDING--------#


day_spending_amount = (
    df_clean[df_clean['Type'] == 'debit']
    .groupby('Day')['Amount']
    .sum()
    .reindex(day_sequence)
)

print("Spending by Day of Week:\n")
print(day_spending_amount)

Spending by Day of Week:

Day
Monday       254099.0
Tuesday      265529.0
Wednesday    307364.0
Thursday     187390.0
Friday       188350.0
Saturday     259715.0
Sunday       216454.0
Name: Amount, dtype: float64


In [107]:
#---------WEEKDAY AND WEEKEND ANALYSIS---------#

weekend_set = ['Saturday', 'Sunday']
weekday_set = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']

weekend_spend = day_spending_amount.loc[weekend_set].sum()
weekday_spend = day_spending_amount.loc[weekday_set].sum()

avg_weekend_total = weekend_spend / len(weekend_set)
avg_weekday_total = weekday_spend / len(weekday_set)

diff = ((avg_weekend_total - avg_weekday_total) / avg_weekday_total) * 100

print("Average Weekday Spend : ₹{:.2f}".format(avg_weekday_total))
print("Average Weekend Spend : ₹{:.2f}".format(avg_weekend_total))
print("Weekend Difference    : {:.2f}%".format(diff))

# Are weekends 50% more expensive than weekdays? Surface this.
# --> No, If we look at the numbers we can see that weekends are a little cheaper than weekdays. They are 1.02 percent cheaper to be exact

Average Weekday Spend : ₹240546.40
Average Weekend Spend : ₹238084.50
Weekend Difference    : -1.02%


In [108]:
#---------DAY OF WEEK SUMMARY---------#

print("=" * 65)
print("           DAY-OF-WEEK SPENDING ANALYSIS")
print("=" * 65)

print("\nHighest Spending Day :", day_spending_amount.idxmax())
print("Amount               : ₹{:.2f}".format(day_spending_amount.max()))

print("\nLowest Spending Day  :", day_spending_amount.idxmin())
print("Amount               : ₹{:.2f}".format(day_spending_amount.min()))

print("\nAverage Weekday Spend : ₹{:.2f}".format(avg_weekday_total))
print("Average Weekend Spend : ₹{:.2f}".format(avg_weekend_total))

if diff > 0:
    print(f"\nWeekends are {diff:.2f}% more expensive than weekdays.")
else:
    print(f"\nWeekends are {abs(diff):.2f}% cheaper than weekdays.")

print("\nInsight:")
print("Rahul spends slightly more during weekdays than weekends.")

print("\n" + "=" * 65)
print("Day-of-Week Analysis Completed Successfully")
print("=" * 65)

           DAY-OF-WEEK SPENDING ANALYSIS

Highest Spending Day : Wednesday
Amount               : ₹307364.00

Lowest Spending Day  : Thursday
Amount               : ₹187390.00

Average Weekday Spend : ₹240546.40
Average Weekend Spend : ₹238084.50

Weekends are 1.02% cheaper than weekdays.

Insight:
Rahul spends slightly more during weekdays than weekends.

Day-of-Week Analysis Completed Successfully


# **FEATURE 7 : ANOMALY DETECTION**

In [109]:
#-----------CATEGORY MEAN AND STANDARD------------#

debit_new_df = df_clean[df_clean['Type'] == 'debit'].copy()

debit_new_df['category_mean'] = debit_new_df.groupby('category')['Amount'].transform('mean')
debit_new_df['category_std'] = debit_new_df.groupby('category')['Amount'].transform('std')

print("Sample of category mean and std:\n")
print(debit_new_df[['category', 'Amount', 'category_mean', 'category_std']].head(10))

Sample of category mean and std:

             category  Amount  category_mean  category_std
0          E-commerce  2462.0    3468.426573   4437.942044
1           Transport    50.0     229.896000    147.398938
3   Personal Transfer  1828.0    1970.355263   3435.128111
4      Quick Commerce   270.0     532.318584    204.991997
5      Quick Commerce   625.0     532.318584    204.991997
6           Transport   148.0     229.896000    147.398938
7      Quick Commerce   482.0     532.318584    204.991997
8       Food Delivery   537.0     438.645604    154.327023
9         Investments  3956.0   10789.565217   5376.350839
10          Transport   350.0     229.896000    147.398938


In [110]:
#----------COMPUTING Z - SCORE------------

debit_new_df['z_score'] = (
    (debit_new_df['Amount'] - debit_new_df['category_mean']) / debit_new_df['category_std']
)

print("Sample z-scores:\n")
print(
    debit_new_df[['Date', 'vendor_cleaned_names', 'category', 'Amount', 'z_score']]
    .head(10)
)

Sample z-scores:

         Date vendor_cleaned_names           category  Amount   z_score
0  2024-01-01               Amazon         E-commerce  2462.0 -0.226778
1  2024-01-01                 BMTC          Transport    50.0 -1.220470
3  2024-01-01         P2P Transfer  Personal Transfer  1828.0 -0.041441
4  2024-01-01              Blinkit     Quick Commerce   270.0 -1.279653
5  2024-01-01                Zepto     Quick Commerce   625.0  0.452122
6  2024-01-01                 Uber          Transport   148.0 -0.555608
7  2024-01-01              Blinkit     Quick Commerce   482.0 -0.245466
8  2024-01-02               Swiggy      Food Delivery   537.0  0.637312
9  2024-01-02                Groww        Investments  3956.0 -1.271042
10 2024-01-02                  Ola          Transport   350.0  0.814823


In [111]:
#---------DETECTING ANOMALIES---------#

anomalies_detect = debit_new_df[debit_new_df['z_score'] > 2].copy()

anomalies_detect = anomalies_detect.sort_values(by='z_score', ascending=False)

print("Number of anomalies detected:", len(anomalies_detect))
print("\nTop 10 anomalies:\n")

print(
    anomalies_detect[['Date', 'vendor_cleaned_names', 'category', 'Amount', 'z_score']]
    .head(10)
)

Number of anomalies detected: 33

Top 10 anomalies:

           Date vendor_cleaned_names           category   Amount   z_score
1079 2024-05-27         P2P Transfer  Personal Transfer  17831.0  4.617192
1130 2024-06-03         P2P Transfer  Personal Transfer  16369.0  4.191589
1298 2024-06-26               Amazon         E-commerce  22008.0  4.177516
269  2024-02-07               Amazon         E-commerce  21986.0  4.172559
475  2024-03-05               Amazon         E-commerce  19917.0  3.706352
984  2024-05-15         P2P Transfer  Personal Transfer  14416.0  3.623051
699  2024-04-05         P2P Transfer  Personal Transfer  14305.0  3.590738
414  2024-02-26           Restaurant        Restaurants   8383.0  3.432801
450  2024-03-02               Amazon         E-commerce  18273.0  3.335910
1291 2024-06-25               Amazon         E-commerce  17504.0  3.162631


In [112]:
#----------FINAL ANOMALY REPORT----------#

print("=" * 70)
print("                    ANOMALY DETECTION REPORT")
print("=" * 70)

print(f"\nTotal Anomalies Found : {len(anomalies_detect)}")

print("\nTop 5 Most Unusual Transactions:\n")

top_5_anomalies_detect = anomalies_detect[
    ['Date', 'vendor_cleaned_names', 'category', 'Amount', 'z_score']
].head(5)

print(top_5_anomalies_detect.to_string(index=False))

print("\nCategory-wise anomaly counts:\n")
print(anomalies_detect['category'].value_counts())

print("\n" + "=" * 70)

                    ANOMALY DETECTION REPORT

Total Anomalies Found : 33

Top 5 Most Unusual Transactions:

      Date vendor_cleaned_names          category  Amount  z_score
2024-05-27         P2P Transfer Personal Transfer 17831.0 4.617192
2024-06-03         P2P Transfer Personal Transfer 16369.0 4.191589
2024-06-26               Amazon        E-commerce 22008.0 4.177516
2024-02-07               Amazon        E-commerce 21986.0 4.172559
2024-03-05               Amazon        E-commerce 19917.0 3.706352

Category-wise anomaly counts:

category
E-commerce           11
Food Delivery         6
Card Payment          5
Personal Transfer     4
Restaurants           3
Entertainment         2
Fuel                  1
Others                1
Name: count, dtype: int64



# **FEATURE 8 : SPENDING ARCHETYPE DETECTION**

In [113]:
#------------PRECOMPUTE ARCHETYPE METRICS----------#

debit_account = df_clean[df_clean['Type'] == 'debit'].copy()

# total debit spend
tot_debit_spend = debit_account['Amount'].sum()

# category spend (absolute rupees)
category_spend = debit_account.groupby('category')['Amount'].sum()

# category share in percentage
category_share = (category_spend / tot_debit_spend) * 100

# vendor spend
vendor_spender = debit_account.groupby('vendor_cleaned_names')['Amount'].sum()

# total transactions
total_transactions = len(debit_account)

# food delivery transactions only
food_delivery_transactions = debit_account[debit_account['category'] == 'Food Delivery']

# late-night food orders: 9 PM onwards + 12 AM–1:59 AM
late_night_food = food_delivery_transactions[
    (food_delivery_transactions['Hour'] >= 21) | (food_delivery_transactions['Hour'] <= 1)
]

print("Archetype metrics prepared successfully.")
print("Total debit spend:", tot_debit_spend)

print("\nCategory Spend:\n")
print(category_spend)

print("\nCategory Share (%):\n")
print(category_share.round(2))

Archetype metrics prepared successfully.
Total debit spend: 1678901.0

Category Spend:

category
Cafe                  25690.0
Card Payment         102074.0
Cash Withdrawal       45500.0
E-commerce           495985.0
Entertainment         19957.0
Food Delivery        159667.0
Fuel                  17895.0
Groceries             49178.0
Investments          248160.0
Others                35428.0
Personal Transfer    149747.0
Quick Commerce        60152.0
Rent                 108000.0
Restaurants           58121.0
Subscriptions          5691.0
Transport             57474.0
Utilities             40182.0
Name: Amount, dtype: float64

Category Share (%):

category
Cafe                  1.53
Card Payment          6.08
Cash Withdrawal       2.71
E-commerce           29.54
Entertainment         1.19
Food Delivery         9.51
Fuel                  1.07
Groceries             2.93
Investments          14.78
Others                2.11
Personal Transfer     8.92
Quick Commerce        3.58
Rent     

In [114]:
#---------ARCHETYPE DETECTION FUNCTIONS----------#

def detect_foodie():
    share = category_share.get('Food Delivery', 0)
    if share >= 9:
        return f"The Foodie -> Food Delivery share = {share:.2f}% of total spend"
    return None


def detect_investor():
    share = category_share.get('Investments', 0)
    if share >= 12:
        return f"The Investor -> Investment share = {share:.2f}% of total spend"
    return None


def detect_late_night_snacker():
    if len(food_delivery_transactions) > 0:
        pct = (len(late_night_food) / len(food_delivery_transactions)) * 100
        if pct >= 20:
            return f"The Late-Night Snacker -> {pct:.2f}% of food orders placed after 9 PM"
    return None


def detect_convenience_lover():
    share = category_share.get('Quick Commerce', 0)
    if share >= 4:
        return f"The Convenience Lover -> Quick Commerce share = {share:.2f}% of total spend"
    return None


def detect_cafe_hopper():
    share = category_share.get('Cafe', 0)
    if share >= 2:
        return f"The Cafe Hopper -> Cafe share = {share:.2f}% of total spend"
    return None


def detect_shopaholic():
    share = category_share.get('E-commerce', 0)
    if share >= 25:
        return f"The Shopaholic -> E-commerce share = {share:.2f}% of total spend"
    return None


def detect_city_commuter():
    share = category_share.get('Transport', 0)
    if share >= 5:
        return f"The City Commuter -> Transport share = {share:.2f}% of total spend"
    return None


def detect_big_spender():
    if tot_debit_spend > 1500000:
        return f"The Big Spender -> Total debit spend = ₹{tot_debit_spend:,.2f}"
    return None


# UNIQUE INVENTED ARCHETYPE
def detect_convenience_maximalist():
    combined = (
        category_share.get('Food Delivery', 0) +
        category_share.get('Quick Commerce', 0) +
        category_share.get('Cafe', 0)
    )
    if combined >= 14:
        return f"The Convenience Maximalist -> Food + Quick Commerce + Cafe = {combined:.2f}% of total spend"
    return None

In [115]:
#-------------RUNNING ARCHETYPE RULES-------------#

archetype_detect_results = []

archetype_detection = [
    detect_foodie,
    detect_investor,
    detect_late_night_snacker,
    detect_convenience_lover,
    detect_cafe_hopper,
    detect_shopaholic,
    detect_city_commuter,
    detect_big_spender,
    detect_convenience_maximalist
]

for rule in archetype_detection:
    result = rule()
    if result is not None:
        archetype_detect_results.append(result)

print("Number of archetypes matched:", len(archetype_detect_results))
print("\nMatched Archetypes:\n")

for item in archetype_detect_results:
    print("-", item)

Number of archetypes matched: 6

Matched Archetypes:

- The Foodie -> Food Delivery share = 9.51% of total spend
- The Investor -> Investment share = 14.78% of total spend
- The Late-Night Snacker -> 20.60% of food orders placed after 9 PM
- The Shopaholic -> E-commerce share = 29.54% of total spend
- The Big Spender -> Total debit spend = ₹1,678,901.00
- The Convenience Maximalist -> Food + Quick Commerce + Cafe = 14.62% of total spend


In [116]:
#-----------ARCHETYPE FINAL REPORT-----------#


print("=" * 78)
print("                        SPENDING ARCHETYPE REPORT")
print("=" * 78)

print(f"\nRahul matches {len(archetype_detect_results)} spending archetypes:\n")

for i, archetype in enumerate(archetype_detect_results, start=1):
    print(f"{i}. {archetype}")

print("\n" + "=" * 78)

                        SPENDING ARCHETYPE REPORT

Rahul matches 6 spending archetypes:

1. The Foodie -> Food Delivery share = 9.51% of total spend
2. The Investor -> Investment share = 14.78% of total spend
3. The Late-Night Snacker -> 20.60% of food orders placed after 9 PM
4. The Shopaholic -> E-commerce share = 29.54% of total spend
5. The Big Spender -> Total debit spend = ₹1,678,901.00
6. The Convenience Maximalist -> Food + Quick Commerce + Cafe = 14.62% of total spend



In [117]:
#----------UNIQUE INVENTED ARCHETYPE EXPLANATION----------#

combined_sharer = (
    category_share.get('Food Delivery', 0) +
    category_share.get('Quick Commerce', 0) +
    category_share.get('Cafe', 0)
)

print("Invented Archetype: The Convenience Maximalist\n")
print("Detection Rule:")
print("Rahul is tagged as a Convenience Maximalist if")
print("Food Delivery % + Quick Commerce % + Cafe % >= 14% of total debit spend.\n")     # RULE USED

print(f"Rahul's combined share = {combined_sharer:.2f}%")

if combined_sharer >= 14:
    print("Result: Rahul matches this archetype.")
else:
    print("Result: Rahul does not match this archetype.")

Invented Archetype: The Convenience Maximalist

Detection Rule:
Rahul is tagged as a Convenience Maximalist if
Food Delivery % + Quick Commerce % + Cafe % >= 14% of total debit spend.

Rahul's combined share = 14.62%
Result: Rahul matches this archetype.


In [118]:
#------FINAL REPORT DATA PREPARATION------#

# Executive summary
total_credits = df_clean[df_clean['Type'] == 'credit']['Amount'].sum()
total_debits = df_clean[df_clean['Type'] == 'debit']['Amount'].sum()
net_savings = total_credits - total_debits
savings_rate = (net_savings / total_credits) * 100 if total_credits != 0 else 0
transaction_count = len(df_clean)
unique_vendors = df_clean['vendor_cleaned_names'].nunique()

# Top categories
top_categories = category_spend.sort_values(ascending=False).head(5)
top_category_share = ((top_categories / total_debits) * 100).round(2)

# Top vendors
top_vendors = debit_account.groupby('vendor_cleaned_names')['Amount'].sum().sort_values(ascending=False).head(5)

vendor_txn_counts = (
    debit_account[debit_account['vendor_cleaned_names'].isin(top_vendors.index)]
    .groupby('vendor_cleaned_names')
    .size()
)

# Peak hour by category from Feature 6
hour_matrix = df_clean.groupby(['category', 'Hour']).size().unstack(fill_value=0)

food_peak_hour = hour_matrix.loc['Food Delivery'].idxmax() if 'Food Delivery' in hour_matrix.index else None
cafe_peak_hour = hour_matrix.loc['Cafe'].idxmax() if 'Cafe' in hour_matrix.index else None
quick_peak_hour = hour_matrix.loc['Quick Commerce'].idxmax() if 'Quick Commerce' in hour_matrix.index else None

# Monthly trend for Food Delivery
food_monthly = monthly_pivot_list.loc['Food Delivery'] if 'Food Delivery' in monthly_pivot_list.index else None

# Top anomalies
top_anomalies = anomalies_detect.sort_values('z_score', ascending=False).head(5)

print("Final report data prepared successfully.")

Final report data prepared successfully.


In [119]:
#------TEXT BAR HELPER FUNCTION------#

def make_bar(value, max_value, bar_length=20):
    if max_value == 0:
        return ""
    filled = int((value / max_value) * bar_length)
    return "#" * filled

In [120]:
#------FINAL SPENDDNA PRINTED REPORT------#

print("=" * 78)
print("SpendDNA REPORT  -  RAHUL SHARMA")
print(f"6 months  -  {transaction_count:,} transactions  -  Jan to Jun 2024")
print("=" * 78)

# EXECUTIVE SUMMARY
print("\nEXECUTIVE SUMMARY")
print(f"Total credits    : ₹{total_credits:,.2f}")
print(f"Total debits     : ₹{total_debits:,.2f}")
print(f"Net change       : ₹{net_savings:,.2f}")

if net_savings < 0:
    print(f"Savings rate     : {savings_rate:.2f}%   (OVERSPENDING)")
else:
    print(f"Savings rate     : {savings_rate:.2f}%   (SAVING)")

print(f"Transactions     : {transaction_count}")
print(f"Unique vendors   : {unique_vendors}")

# TOP CATEGORIES
print("\nTOP CATEGORIES (% of debit total)")
max_cat_value = top_categories.max()

for cat, amount in top_categories.items():
    share = top_category_share[cat]
    bar = make_bar(amount, max_cat_value, bar_length=18)
    print(f"{cat:<18} {bar:<18} {share:>6.2f}%   ₹{amount:,.2f}")

# TOP VENDORS
print("\nTOP VENDORS")
for vendor, amount in top_vendors.items():
    txn_count = vendor_txn_counts.get(vendor, 0)
    print(f"{vendor:<18} ₹{amount:>10,.2f}   ({txn_count} orders)")

# TIME-OF-DAY PATTERNS
print("\nTIME-OF-DAY PATTERNS")
if food_peak_hour is not None:
    print(f"Food Delivery peaks : {food_peak_hour:02d}:00")
if cafe_peak_hour is not None:
    print(f"Cafe peaks          : {cafe_peak_hour:02d}:00")
if quick_peak_hour is not None:
    print(f"Quick Commerce peak : {quick_peak_hour:02d}:00")

# MONTHLY TREND
print("\nMONTHLY TREND (Food Delivery)")
if food_monthly is not None:
    max_food = food_monthly.max()
    for month, value in food_monthly.items():
        bar = make_bar(value, max_food, bar_length=15)
        print(f"{month:<3} ₹{value:>8,.2f}  {bar}")

# TOP ANOMALIES
print("\nTOP ANOMALIES")
for _, row in top_anomalies.iterrows():
    print(f"{row['Date'].strftime('%d %b')} - {row['vendor_cleaned_names']:<15} ₹{row['Amount']:>8,.2f}   (z={row['z_score']:.2f})")

print("=" * 78)

SpendDNA REPORT  -  RAHUL SHARMA
6 months  -  1,310 transactions  -  Jan to Jun 2024

EXECUTIVE SUMMARY
Total credits    : ₹509,774.00
Total debits     : ₹1,678,901.00
Net change       : ₹-1,169,127.00
Savings rate     : -229.34%   (OVERSPENDING)
Transactions     : 1310
Unique vendors   : 38

TOP CATEGORIES (% of debit total)
E-commerce         ##################  29.54%   ₹495,985.00
Investments        #########           14.78%   ₹248,160.00
Food Delivery      #####                9.51%   ₹159,667.00
Personal Transfer  #####                8.92%   ₹149,747.00
Rent               ###                  6.43%   ₹108,000.00

TOP VENDORS
Amazon             ₹299,462.00   (70 orders)
Zerodha            ₹210,000.00   (14 orders)
P2P Transfer       ₹149,747.00   (76 orders)
Rent               ₹108,000.00   (6 orders)
Swiggy             ₹104,351.00   (243 orders)

TIME-OF-DAY PATTERNS
Food Delivery peaks : 20:00
Cafe peaks          : 10:00
Quick Commerce peak : 19:00

MONTHLY TREND (Food Deliver

In [122]:
#------FINAL ARCHETYPES------#

print("\nSPENDING ARCHETYPES")
for i, archetype in enumerate(archetype_detect_results, start=1):
    print(f"{i}. {archetype}")

print("\nFINAL VERDICT")

# financial health status
if savings_rate < 0:
    print("Rahul is currently an over-spender because total debits exceed total credits by a large margin.")
else:
    print("Rahul is currently saving overall, with credits higher than debits.")

# strongest behavioural signals
top_category = top_categories.idxmax()
top_vendor = top_vendors.idxmax()

print(f"His strongest spending signal is {top_category}, with the highest category spend of ₹{top_categories.max():,.2f}.")
print(f"The most expensive vendor relationship is with {top_vendor}, where total spend reached ₹{top_vendors.max():,.2f}.")

# late-night / lifestyle signal
foodie_match = any("Late-Night Snacker" in a for a in archetype_detect_results)
if foodie_match:
    print("Food delivery behaviour also shows a noticeable late-night ordering pattern, indicating convenience-led spending habits.")

# investor signal
investor_match = any("Investor" in a for a in archetype_detect_results)
if investor_match:
    print("At the same time, Rahul consistently invests a meaningful share of his spending, showing that he is not purely impulsive in money behaviour.")

# custom archetype interpretation
custom_match = any("Convenience Maximalist" in a for a in archetype_detect_results)
if custom_match:
    print("The custom archetype 'Convenience Maximalist' suggests a lifestyle heavily driven by convenience spending across food delivery, quick commerce, and cafes.")

print("\nOverall, Rahul’s financial profile reflects a mix of high discretionary spending, strong e-commerce behaviour, and regular investing, but his current spending level is not sustainable without higher income or tighter budget control.")
print("=" * 78)


SPENDING ARCHETYPES
1. The Foodie -> Food Delivery share = 9.51% of total spend
2. The Investor -> Investment share = 14.78% of total spend
3. The Late-Night Snacker -> 20.60% of food orders placed after 9 PM
4. The Shopaholic -> E-commerce share = 29.54% of total spend
5. The Big Spender -> Total debit spend = ₹1,678,901.00
6. The Convenience Maximalist -> Food + Quick Commerce + Cafe = 14.62% of total spend

FINAL VERDICT
Rahul is currently an over-spender because total debits exceed total credits by a large margin.
His strongest spending signal is E-commerce, with the highest category spend of ₹495,985.00.
The most expensive vendor relationship is with Amazon, where total spend reached ₹299,462.00.
Food delivery behaviour also shows a noticeable late-night ordering pattern, indicating convenience-led spending habits.
At the same time, Rahul consistently invests a meaningful share of his spending, showing that he is not purely impulsive in money behaviour.
The custom archetype 'Conv